In [1]:
# test ML model
# create class that handles RF implementation
import sys
from pathlib import Path
import pandas as pd

# notebook directory
notebook_dir = Path().resolve()

# project root = one level up from notes/
project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from elastic_net.elastic_net import ElasticNetModel
import json

tickers_path = project_root / "data" / "tickers.json"

with open(tickers_path, "r") as file:
    ticker_dict = json.load(file)

stable_start = "2015-01-01" 
stable_end = "2018-12-31" 
unstable_start = "2019-01-01" 
unstable_end = "2022-12-31" 

time_dict = {"stable" : [stable_start, stable_end], "unstable" : [unstable_start, unstable_end], "full" : [stable_start, unstable_end]}

all_tickers = []
window = 21
target_window = 5

results = pd.DataFrame(columns=["sector", "time", "rmse", "mse", "mae", "r2", "mape", "rel_mse"])

In [7]:
def test_model(start_date, end_date):
    try:
        model = ElasticNetModel(tickers, window, start_date, end_date, target_window=target_window)
    
        model.fit_model()
    except Exception as e:
        print("err: ", e)
        raise e

    metrics = model.test_results()

    for k, v in metrics.items():
        print(f"{k} = {v}")
    return metrics

In [13]:
for sector, tickers in ticker_dict.items():
    # train model per sector 
    # compare with overall model for all sectors
    # compare with different data lengths
    print(f"\n\n----- Sector: {sector} -----\n")
    for descr, time in time_dict.items():
        [start, end] = time
        print(f"\n {descr} Time Period {start} to {end}\n")
        try:
            res = test_model(start, end)
            results = pd.concat([
                results,
                pd.DataFrame([{
                    "sector": sector,
                    "time" : descr,
                    "rmse": res["rmse"],
                    "mse": res["mse"],
                    "mae": res["mae"],
                    "r2": res["r2"],
                    "mape": res["mape"]
                }])
            ], ignore_index=True)
            all_tickers += tickers
        except:
            print("skipping...")

print(f"\n\n----- All Tickers -----\n")
for descr, time in time_dict.items():
    [start, end] = time
    print(f"\n {descr} Time Period {start} to {end}\n")
    res = test_model(start, end)
    results = pd.concat([
        results,
        pd.DataFrame([{
            "sector": "all_tickers",
            "time" : descr,
            "rmse": res["rmse"],
            "mse": res["mse"],
            "mae": res["mae"],
            "r2": res["r2"],
            "mape": res["mape"]
        }])
    ], ignore_index=True)



----- Sector: Real Estate -----


 stable Time Period 2015-01-01 to 2018-12-31



[*********************100%***********************]  31 of 31 completed
C:\Users\Owner\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


mse = 0.007822172540348019
rmse = 0.0884430468739517
mae = 0.06588013294190424
r2 = 0.11656188081265595
mape = 0.4327590318818275
rel_mse = 0.8834381191873439

 unstable Time Period 2019-01-01 to 2022-12-31



[*********************100%***********************]  31 of 31 completed
C:\Users\Owner\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


mse = 0.018854008647671037
rmse = 0.13730990003517968
mae = 0.10309979270039017
r2 = -0.163217331315322
mape = 0.4209256713938499
rel_mse = 1.163217331315322

 full Time Period 2015-01-01 to 2022-12-31



...

mse = 0.02534262554892254
rmse = 0.15919367308069293
mae = 0.10177567267531033
r2 = 0.1729547615885928
mape = 509382359220.71106
rel_mse = 0.8270452384114072

 unstable Time Period 2019-01-01 to 2022-12-31



[*********************100%***********************]  55 of 55 completed
C:\Users\Owner\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


mse = 0.038017501445541606
rmse = 0.19498077198929542
mae = 0.13528856502100842
r2 = 0.2429502927117978
mape = 0.41971870051450044
rel_mse = 0.7570497072882021

 full Time Period 2015-01-01 to 2022-12-31



[*********************100%***********************]  55 of 55 completed
C:\Users\Owner\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\Owner\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


mse = 0.029902621643782337
rmse = 0.172923745170472
mae = 0.11940080247258644
r2 = 0.3090765624415218
mape = 86761623855.60556
rel_mse = 0.6909234375584783


In [14]:
results.head()

,sector,time,rmse,mse,mae,r2,mape,rel_mse
0,Real Estate,stable,0.088443,0.007822,0.065880,0.116562,0.432760,NaN
1,Real Estate,unstable,0.137310,0.018854,0.103100,-0.163218,0.420926,NaN
2,Real Estate,full,0.122038,0.014893,0.090094,0.023805,0.423995,NaN
3,Basic Materials,stable,0.124519,0.015505,0.087278,0.171818,0.432412,NaN
4,Basic Materials,unstable,0.160929,0.025898,0.121584,0.077545,0.411734,NaN


In [15]:
results.groupby("sector")["rmse"].mean()

sector
Basic Materials           0.143045
Communication Services    0.189390
Consumer Cyclical         0.175700
Consumer Defensive        0.135952
Energy                    0.155831
Financial Services        0.127559
Healthcare                0.152758
Industrials               0.140810
Real Estate               0.115930
Technology                0.177679
Utilities                 0.128391
all_tickers               0.175699
Name: rmse, dtype: float64

In [16]:
results.groupby("time")["rmse"].mean()

time
full        0.148987
stable      0.139452
unstable    0.166247
Name: rmse, dtype: float64